# RAG를 위한 PREPROCESS2

## Splitter

1. 토큰 제한 회피: 대부분의 LLM은 입력으로 받을 수 있는 **최대 토큰 수(컨텍스트 창)**에 제한이 있습니다. 긴 문서를 통째로 넣으려 하면 오류가 발생하거나, 문서의 중요한 부분이 잘릴 수 있습니다.
2. 임베딩 효율성: 텍스트를 벡터 저장소에 임베딩할 때, 너무 큰 청크는 특정 질문에 대한 정확한 의미 정보를 희석시킬 수 있습니다. 적절한 크기로 분할해야 검색 정확도가 높아집니다.
3. 처리 속도 및 비용: 작은 청크를 사용하면 LLM이 처리할 데이터의 양이 줄어들어 속도가 빨라지고 비용이 절감됩니다.

**텍스트 분할기(Text Splitter)**는 방대한 텍스트 문서나 데이터를 관리하고 처리하기 위한 핵심 도구이다. 이 객체의 주된 목적은 긴 텍스트를 **더 작고 관리하기 쉬운 청크(Chunk)**로 나누는 것

(토큰이랑 다르다. 청크는 RAG, 검색, 요약 같은 작업을 위해 입력 텍스트를 나누는 단위이다. 청크는 덩어리임 토큰은 모델이 텍스트를 읽는 최소단위고!)

> 결국 쪼개는 것이다. 긴 문서를 통째로 넣으면 문제가 있다. **LLM이 감당할 수 있는 컨텍스트 length보다 많으면 동작하지 않기 때문에 문서를 쪼개는 것이다.**  (max toekn error 방지 위해 문서를 청크로 나눈다)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_text_splitters import CharacterTextSplitter, MarkdownTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# chunk_size: 생성될 각 청크의 최대 크기를 정의합니다. (일반적으로 문자 수 또는 토큰 수 기준)
# chunk_overlap: 연속된 청크 간에 겹치는 텍스트의 양을 정의합니다.
# 목적: 청크 경계에서 중요한 의미나 문맥이 단절되는 것을 방지하기 위해 사용됩니다. 적절한 오버랩을 설정하면, 한 청크의 끝과 다음 청크의 시작 부분이 문맥을 이어갈 수 있습니다.

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=50,
    length_function=len,
    is_separator_regex=False,
)

FILE_PATH = "./SPRi AI Brief_6월호_산업동향_F.pdf"
loader = PyPDFLoader(FILE_PATH)
documents = loader.load()

chunks = text_splitter.split_documents(documents)
print(f"총 {len(chunks)}개 청크 생성")
print(chunks[0].page_content[:200])
print(chunks[0].metadata) 


  chunk_size=250

  한 조각의 최대 길이입니다.
  여기서는 한 chunk를 대략 250자 이하로 자르겠다는 뜻입니다.

  일반 기준:

  - 짧은 질의응답: 300~700
  - 일반 문서 RAG: 500~1000
  - 긴 문맥이 필요한 문서: 1000~2000
  - 너무 작으면 문맥이 끊기고, 너무 크면 검색 정확도가 떨어질 수 있음

  ———

  chunk_overlap=50

  chunk끼리 겹치게 할 문자 수입니다.
  앞 chunk의 끝부분 50자를 다음 chunk에도 포함합니다.

  이유는 문장이 chunk 경계에서 잘려도 문맥이 이어지게 하기 위해서입니다.

  일반 기준:

  - 보통 chunk_size의 10~20%
  - 예: chunk_size=500이면 overlap=50~100
  - 너무 크면 중복 저장이 많아지고 비용이 늘어남

  ———

  length_function=len

  길이를 어떻게 계산할지 정합니다.

  len

  은 파이썬 기본 길이 계산 함수입니다.
  즉, 문자열의 문자 수를 기준으로 chunk 크기를 계산합니다.

  일반 기준:

  - 간단히 쓸 때는 len
  - 모델 토큰 수 기준으로 정확히 자르고 싶으면 토큰 기반 length function 사용

  ———

  is_separator_regex=False

  구분자를 정규표현식으로 해석할지 여부입니다.

  False이면 구분자를 일반 문자열로 봅니다.
  보통 기본값처럼 False를 많이 씁니다.

  예:

  "\n\n"

  을 진짜 줄바꿈 문자열로 봅니다.

  True이면 정규표현식 패턴으로 해석합니다.

  ———

In [ ]:
def create_production_splitter():
    return RecursiveCharacterTextSplitter(
        chunk_size=1200,           # BGE, text-embedding-3-large 최적
        chunk_overlap=250,         # 문맥 연결 완벽
        length_function=len,
        separators=["\n\n", "\n", ". ", "? ", "! ", " ", ""],
        add_start_index=True,
        strip_whitespace=True,
    )

# 사용법
splitter = create_production_splitter()
chunks = splitter.split_documents(documents)

# 메타데이터 확인 (출처 인용용)
for i, chunk in enumerate(chunks[:3]):
    print(f"청크 {i+1}: {len(chunk.page_content)}자")
    print(f"출처: {chunk.metadata.get('source')} 페이지: {chunk.metadata.get('page')}")
    print(f"시작 위치: {chunk.metadata.get('start_index')}")
    print("-" * 50)

예시 결과
```python
청크 1: 22자
출처: ./data/SPRI AI Brief_6월호_산업동향_F.pdf 페이지 0
시작 위치: 0
--------------------------------------------------
청크 2: 878자
출처: ./data/SPRI AI Brief_6월호_산업동향_F.pdf 페이지 1
시작 위치: 0
--------------------------------------------------
청크 3: 19자
출처: ./data/SPRI AI Brief_6월호_산업동향_F.pdf 페이지 2
시작 위치: 0
--------------------------------------------------
```

## VectorStore

Chorma DB 또는 Fassis를 사용해서 벡터 스토어를 만들 수 있다! (크로마와 파이쓰는 랭체인과 연결이 쉽다)

In [ ]:
pip install langchain langchain-google-genai langchain-community 
pip install chromadb faiss-cpu pymilvus 

Chroma는 파이썬 패키지로 쉽게 설치하고 사용할 수 있으며, 로컬에서 실행하거나 서버로 배포할 수 있는 완전한 기능을 갖춘 벡터 데이터베이스입니다. LangChain과의 연동이 매우 쉽습니다.

🛠️ 특징
설치 용이: Python pip 설치 후 바로 사용 가능합니다.

유연성: 영구 저장소(로컬 파일 시스템) 또는 인메모리(RAM) 모드를 모두 지원합니다.

메타데이터 필터링: 메타데이터를 기반으로 검색 결과를 필터링하는 기능이 강력합니다.